In [ ]:
import glob
import pandas as pd
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from numpy.testing.print_coercion_tables import print_new_cast_table
from scipy.stats import gmean
import scipy.stats as stats
import re
import os

from functools import reduce
from matplotlib.colors import to_rgb, to_hex
import colorsys

from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

In [ ]:
def safe_gmean(x):
    x = np.array(x)
    x = x[x > 0]  # only positive values
    return gmean(x) if len(x) > 0 else np.nan

# Helper to classify size
def size_category(size):
    if size < 8000:
        return "Small"
    elif size <= 15000:
        return "Middle"
    else:
        return "Large"
    
palette1 = {
    'MeDiH-MM - dyn_vs_base': 'lightblue',
    'MeDiH-MM - stat_vs_base': 'darkblue',
    'MeDiH-BL - dyn_vs_base': 'orange',
    'MeDiH-BL - stat_vs_base': 'darkorange',
    'MeDiH-BLC - dyn_vs_base': 'lightgreen',
    'MeDiH-BLC - stat_vs_base': 'darkgreen',
    'MeDiH-MM - internal': 'blue',  
    'MeDiH-BL - internal': 'orange',   
    'MeDiH-BLC - internal': 'green',
}


palette = {
    # 1. MeDiH-MM: Blue Family
    'MeDiH-MM - dyn_vs_base': 'dodgerblue',    # Vivid, clear blue
    'MeDiH-MM - stat_vs_base': 'darkblue',         # Very dark contrast
    'MeDiH-MM - internal': 'dodgerblue',

    # 2. MeDiH-BL: Orange Family (Fixed your contrast issue)
    'MeDiH-BL - dyn_vs_base': 'gold',        # Classic bright orange
    'MeDiH-BL - stat_vs_base': 'darkgoldenrod',  # Deep brown-orange (huge contrast)
    'MeDiH-BL - internal': 'gold',

    # 3. MeDiH-BLC: Green Family
    'MeDiH-BLC - dyn_vs_base': 'limegreen',  # Or 'limegreen' for high visibility
    'MeDiH-BLC - stat_vs_base': 'darkgreen',   # Deep forest green
    'MeDiH-BLC - internal': 'limegreen',

    # 4. MeDiH-L: Red Family
    'MeDiH-L - dyn_vs_base': 'tomato',         # Bright "light" red
    'MeDiH-L - stat_vs_base': 'firebrick',        # Deep, scholarly dark red
    'MeDiH-L - internal': 'tomato',

    # 5. MeDiH-BL-R: Purple Family
    'MeDiH-BL-R - dyn_vs_base': 'mediumorchid', # Rich, vibrant purple
    'MeDiH-BL-R - stat_vs_base': 'darkmagenta',       # Deepest purple/black
    'MeDiH-BL-R - internal': 'mediumorchid',
}

palette_params1 = {
    'MeDiH-BL - dyn_vs_base': '#8172B3',
    'MeDiH-BL - stat_vs_base': '#64B5CD',
    'MeDiH-BLC - dyn_vs_base': '#C44E52',
    'MeDiH-BLC - stat_vs_base': '#8C564B',
    'MeDiH-MM - dyn_vs_base': '#7F7F7F',
    'MeDiH-MM - stat_vs_base': '#CCB974'
}

palette_params = {
    'dyn_vs_base-Small': '#C44E52',       # soft red
    'stat_vs_base-Small': '#8C2D31',      # deep red

    'dyn_vs_base-Middle': '#8172B3',      # lavender
    'stat_vs_base-Middle': '#5B4C94',     # dark violet

    'dyn_vs_base-Large': '#CCB974',       # golden beige
    'stat_vs_base-Large': '#9A853F'       # muted olive/brown
}


label_map = {
    "MeDiH-BL - stat_vs_base": "Offline MeDiH-BL (Baseline)",
    "MeDiH-BL - dyn_vs_base": "Online MeDiH-BL",
    "MeDiH-BLC - stat_vs_base": "Offline MeDiH-BLC",
    "MeDiH-BLC - dyn_vs_base": "Online MeDiH-BLC",
    "MeDiH-MM - stat_vs_base": "Offline MeDiH-MM",
    "MeDiH-MM - dyn_vs_base": "Online MeDiH-MM" ,
     "MeDiH-L - stat_vs_base": "Offline MeDiH-L",
    "MeDiH-L - dyn_vs_base": "Online MeDiH-L" ,
     "MeDiH-BL-R - stat_vs_base": "Offline MeDiH-BL-R",
    "MeDiH-BL-R - dyn_vs_base": "Online MeDiH-BL-R" ,
    
    "MeDiH-BL - internal": "Internal MeDiH-BL",
    "MeDiH-BLC - internal": "Internal MeDiH-BLC",  
    "MeDiH-MM - internal": "Internal MeDiH-MM",
    "dyn_vs_base-Large": "Dynamic, Large",
    "dyn_vs_base-Middle": "Dynamic, Middle",
    "dyn_vs_base-Small": "Dynamic, Small",
    "stat_vs_base-Large": "Static, Large",
    "stat_vs_base-Middle": "Static, Middle",
    "stat_vs_base-Small": "Static, Small"
    
}

def read_dfs(pathR, patternR, numGroups=2):
    dfs = {}
    
    for fname in glob.glob(pathR):
       # print(fname)
        df = pd.read_csv(fname, delimiter=' ', header=0, on_bad_lines="skip")
        basename = os.path.basename(fname)
        match = re.search(patternR, basename)
        #print(basename, match)
        
        if match:
            algorithm = match.group(1)
            variant = match.group(numGroups)
            dfs[(algorithm, variant)] = df
        #else:
            #print("No match found. " + basename)
    
    return dfs

In [ ]:
def merge_correct_columns(dfsOurVar, lbs):
    renamed_dfs = []
    for df, label in zip(dfsOurVar, lbs):
        # Rename selected columns       
        renamed = df[['wf_name', 'inp_size', 'ms_1', 'ms_2', 'ms_3']].copy()
        renamed = renamed.rename(columns={
            'ms_1': f'ms_1_{label}',
            'ms_2': f'ms_2_{label}',
            'ms_3': f'ms_3_{label}',
        })    
        renamed_dfs.append(renamed)


    #print(renamed_dfs)
    # Merge them all on 'wf_name'   
    merged_df = reduce(lambda left, right: pd.merge(left, right, on=['wf_name', 'inp_size']), renamed_dfs)
   # print("MERGED:", merged_df)

    merged_df['size'] = merged_df['wf_name'].str.extract(r'_(\d+)\.')[0].fillna("100")
    merged_df['size'] = merged_df['size'].astype(int)


    for col in merged_df.columns:
        if col.startswith("ms_"):
            merged_df[col] = pd.to_numeric(merged_df[col], errors="coerce")
    
    return merged_df

In [ ]:
def size_category(size):
    if size <= 200:
        return 'small'
    elif 1000 <= size <= 8000:
        return 'middle'
    elif 10000 <= size <= 18000:
        return 'large'
    elif 20000 <= size <= 30000:
        return 'largest'
    else:
        return 'other'

size_order = ['small', 'middle', 'large', 'largest']

In [ ]:
algos = ['A1', 'A2', 'A3', 'A4', 'A5']
algo_aliases = {
        'A1': 'MeDiH-BL',
        'A2': 'MeDiH-BLC',
        'A3': 'MeDiH-MM',
        'A4': 'MeDiH-L',
        'A5': 'MeDiH-BL-R'
    }

def buld_plot_df(merged_df):
    # For storing all ratios in long format
    ratios = []

    for algo in algos:
        df = merged_df.copy()
        df['size'] = df['size']  # group key

        # Compute the 3 relations
        df['internal'] = df[f'ms_1_{algo}'] / df[f'ms_2_{algo}']
        df['dyn_vs_base'] =  df['ms_2_A1'] /df[f'ms_1_{algo}']
        df['stat_vs_base'] =  df['ms_2_A1'] / df[f'ms_2_{algo}']
       

        # Reshape to long format for seaborn
        melted = df[['size', 'wf_name', 'internal','inp_size', 'dyn_vs_base', 'stat_vs_base']].melt(#
            id_vars=['size', 'wf_name','inp_size'],
            var_name='relation',
            value_name='ratio'
        )
        melted['algorithm'] = algo

        ratios.append(melted)

    # Concatenate all into one DataFrame
    pl_df = pd.concat(ratios, ignore_index=True)

  

    pl_df['algorithm'] = pl_df['algorithm'].replace(algo_aliases)
    return pl_df 

In [ ]:

#palette.update({
#    # MeDiH-L (Red/DarkRed theme)
#    'MeDiH-L - dyn_vs_base': 'tomato',
#    'MeDiH-L - stat_vs_base': 'firebrick',
#    'MeDiH-L - internal': 'tomato',
    
    # MeDiH-BL-R (Purple/Indigo theme)#
#    'MeDiH-BL-R - dyn_vs_base': 'mediumpurple',
#    'MeDiH-BL-R - stat_vs_base': 'rebeccapurple',
#    'MeDiH-BL-R - internal': 'mediumpurple',
#})
def plotDynAndStaticVsBaseNoInset(our_df, deviationsText, titleText, relations):
   
    filtered_df = our_df[our_df['relation'].isin(relations)].copy()# ['internal']
    filtered_df['alg_rel'] = filtered_df['algorithm'] + ' - ' + filtered_df['relation']
    #print("filtered_df\n", filtered_df)

    fig, ax = plt.subplots(figsize=(24, 12))
    ax = sns.barplot(
        data=filtered_df,
        x='size',
        y='ratio',
        hue='alg_rel',
        estimator=gmean,
        palette=palette,
        ci=None
    )
    if "internal" in relations:
        baseline_line = plt.axhline(y=1, color='red', linestyle='--', linewidth=1.5, label='Equal')    
    plt.xticks(rotation=45)
  
    plt.yticks(fontsize=32)
    plt.xticks(fontsize=32)
    plt.xlabel('Workflow size', fontsize=32)
    plt.ylabel('Improvement over the baseline, times', fontsize=32)
    #plt.legend(fontsize=32, title_fontsize=32)  

    ax.set_ylim(0.75, 2.4)

    
    xticks = ax.get_xticks()
    xticklabels = [tick.get_text() for tick in ax.get_xticklabels()]

    # Replace label "100" with "<100"
    new_labels = ["<100" if lbl == "100" else lbl for lbl in xticklabels]

    # Apply the modified labels
    ax.set_xticklabels(new_labels, fontsize=32, rotation=45, ha='right')

    handles, labels = ax.get_legend_handles_labels()

    # Apply the mapping
    pretty_labels = [label_map.get(lbl, lbl) for lbl in labels]
    
    baseline_line = plt.axhline(y=1, color='red', linestyle='--', linewidth=1.5, label='Baseline: Offline MeDiH-BL')  

    
    # Set updated legend
    ax.legend(handles, pretty_labels, title="Algorithm Variant", fontsize=20, title_fontsize=22, ncol=3)
   # plt.title(f'{titleText} {deviationsText}', fontsize=28) #Makespan improvement over the baseline,

  
    plt.savefig(f'improv_ov_baseline_newest_{deviationsText}.png', facecolor='white',  bbox_inches='tight')
    plt.show()

    grouped = filtered_df.groupby(['size', 'alg_rel'])['ratio'].agg(safe_gmean).reset_index()
    # Print the values used for bar heights
    print(grouped)

    gr = grouped.groupby(['alg_rel'])['ratio'].agg(safe_gmean)
    print(gr)